# Notebook A of 2. Encoder embeddings for the seed study (GPU)

**What this notebook does.** It downloads each dataset, runs the frozen encoder, and saves the raw embeddings
(before PCA) to Drive, one file per configuration and seed. It runs no quantum computation. Notebook B
(`seeds_B_kernels.ipynb`) reads these files and does everything else.

**Why it is split.** Encoding is the slow part and needs a GPU; the quantum kernels now take seconds and need
neither a GPU nor the network. Splitting means a crash or a session timeout during encoding never touches the
kernel stage, and encoding is never repeated once a file exists.

**Runtime.** Use a GPU runtime (Runtime > Change runtime type > T4 GPU). About 30 to 60 minutes in total, most of
it dataset downloads. Resumable: any file already on Drive is skipped.

**Seeds.** 42 (the main run, needed by Notebook B to prove the pipeline reproduces the paper), 7, and 123.

**Exactness.** Every loader and encoder call is copied from the notebook that produced the paper's seed-42 results,
with the literal `42` replaced by `seed`. The original notebook for each configuration:

| Tag | Original notebook | Pool | Encoded |
|---|---|---|---|
| `sst2_bert` | `week2_embedding_pipeline_n1000.ipynb` | 1100/250 by `shuffle(seed)` | once per seed |
| `agnews_roberta` | `week2_agnews_roberta.ipynb` | 1100/250 by `shuffle(seed)` | once per seed |
| `cifar10` | `week2c_cifar10_clip_autosave.ipynb` | fixed, first 550/125 per class in dataset order | **once**, reordered per seed |
| `fashion_vit` | `vision_vit_two_configs.ipynb` | 500/100 per class by `RandomState(seed)` | once per seed |
| `mscoco_flava` | `multimodal_flava_two_configs.ipynb` | fixed, first 730 images in dataset order | **once**, re-paired per seed |
| `snlive_siglip` | `week2_siglip_joint_DRIVE.ipynb` | fixed, first 400/90 per class in dataset order | **once**, same for every seed |

For the three fixed-pool configurations, the seed changes only how the pool is ordered or paired, and that is
applied to the cached embeddings here exactly as the original code applied it to the images (details in the
loader comments). The saved file for every (configuration, seed) is therefore what the original pipeline would
have produced just before its PCA step.

**Output.** `<tag>_seed<seed>_raw.npz` in `qml_project` on Drive, with arrays `E_tr`, `y_tr`, `E_te`, `y_te`.

**Contents.**
0. Setup (runtime check, Drive, helpers)
1. Encoders (one function per model)
2. Dataset loaders and how each one uses the seed
3. Run all configurations
4. Inventory of what is now on Drive
5. Troubleshooting and hand-off to Notebook B

---
# Section 0. Setup

Mounts Drive, confirms a GPU is present, and defines where files go.

In [ ]:
!pip install datasets transformers torch torchvision scikit-learn numpy --quiet
import numpy as np, torch, time, os, random, warnings
from PIL import Image
from datasets import load_dataset
import transformers
from transformers import (AutoTokenizer, AutoModel, AutoProcessor, AutoImageProcessor,
                          CLIPModel, CLIPProcessor, FlavaProcessor, FlavaModel)
warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# The original run used exact fp32 and PIL-based image processors. Newer torch enables TF32 convolutions on
# Ampere GPUs and newer transformers defaults to torch-based 'fast' image processors; both change image
# embeddings slightly (a lot for 28x28 and 32x32 inputs upsampled to 224). Restore the original behaviour.
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False
print('device:', DEVICE, '| torch', torch.__version__, '| transformers', transformers.__version__, '| TF32 off')
if DEVICE == 'cpu': print('WARNING: no GPU. Encoding will be very slow. Switch to a GPU runtime.')

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/qml_project'; os.makedirs(SAVE_DIR, exist_ok=True)
raw_path = lambda tag, seed: os.path.join(SAVE_DIR, f'{tag}_seed{seed}_raw.npz')
def save_raw(tag, seed, E_tr, y_tr, E_te, y_te):
    np.savez(raw_path(tag, seed), E_tr=E_tr, y_tr=y_tr, E_te=E_te, y_te=y_te)
    print(f'  saved {tag} seed {seed}: E_tr {E_tr.shape}, E_te {E_te.shape}')

SEEDS = [42, 7, 123]

# Configurations whose raw files should be deleted and re-encoded in this run. Use this after Notebook B reports
# FAIL for a configuration and the cause has been fixed here. Leave empty to keep everything already on Drive.
REDO = []
for _t in REDO:
    for _s in SEEDS:
        if os.path.exists(raw_path(_t, _s)): os.remove(raw_path(_t, _s)); print('deleted', raw_path(_t, _s))

def l2(h): return h / torch.linalg.norm(h, dim=-1, keepdim=True)
_cache = {}

---
# Section 1. Encoders

Each function returns L2-normalized embeddings where the original normalized, and raw first-token states for the
masked-LM family where the original did not. CLIP and SigLIP are computed from the model's parts because recent
`transformers` releases changed `get_image_features` and `get_text_features` to return an output object; the
values are what the original calls returned.

In [ ]:
def bert_cls(name, texts, batch=32):
    if name not in _cache:
        _cache[name] = (AutoTokenizer.from_pretrained(name), AutoModel.from_pretrained(name).to(DEVICE).eval())
    tok, mdl = _cache[name]; out = []
    for i in range(0, len(texts), batch):
        inp = tok(texts[i:i+batch], padding=True, truncation=True, max_length=128, return_tensors='pt').to(DEVICE)
        with torch.no_grad(): out.append(mdl(**inp).last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(out)

def clip_images(images, batch=64):
    if 'clip' not in _cache:
        _cache['clip'] = (CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32', use_fast=False),
                          CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE).eval())
    proc, mdl = _cache['clip']; out = []
    for i in range(0, len(images), batch):
        inp = proc(images=images[i:i+batch], return_tensors='pt', padding=True).to(DEVICE)
        with torch.no_grad():   # pooled vision output -> visual projection (512-dim), as get_image_features
            out.append(l2(mdl.visual_projection(mdl.vision_model(pixel_values=inp['pixel_values']).pooler_output)).cpu().numpy())
    return np.vstack(out)

def vit_images(images, batch=64):
    if 'vit' not in _cache:
        _cache['vit'] = (AutoImageProcessor.from_pretrained('google/vit-base-patch16-224', use_fast=False),
                         AutoModel.from_pretrained('google/vit-base-patch16-224').to(DEVICE).eval())
    proc, mdl = _cache['vit']; out = []
    for i in range(0, len(images), batch):
        inp = proc(images=[im.convert('RGB') for im in images[i:i+batch]], return_tensors='pt').to(DEVICE)
        with torch.no_grad(): out.append(l2(mdl(**inp).last_hidden_state[:, 0, :]).cpu().numpy())
    return np.vstack(out)

def flava_towers(images, texts, batch=32):
    if 'flava' not in _cache:
        _cache['flava'] = (FlavaProcessor.from_pretrained('facebook/flava-full', use_fast=False),
                           FlavaModel.from_pretrained('facebook/flava-full').to(DEVICE).eval())
    proc, mdl = _cache['flava']; ei, et = [], []
    with torch.no_grad():
        for i in range(0, len(images), batch):
            inp = proc(images=images[i:i+batch], return_tensors='pt').to(DEVICE)
            ei.append(l2(mdl.get_image_features(pixel_values=inp['pixel_values']).last_hidden_state[:, 0, :]).cpu().numpy())
        for i in range(0, len(texts), batch):
            inp = proc(text=list(texts[i:i+batch]), return_tensors='pt', padding=True, truncation=True, max_length=77).to(DEVICE)
            et.append(l2(mdl.get_text_features(input_ids=inp['input_ids'], attention_mask=inp['attention_mask']).last_hidden_state[:, 0, :]).cpu().numpy())
    return np.vstack(ei), np.vstack(et)

def siglip_towers(images, texts, batch=32):
    if 'siglip' not in _cache:
        _cache['siglip'] = (AutoProcessor.from_pretrained('google/siglip-base-patch16-224', use_fast=False),
                            AutoModel.from_pretrained('google/siglip-base-patch16-224').to(DEVICE).eval())
    proc, mdl = _cache['siglip']; ei, et = [], []
    with torch.no_grad():
        for i in range(0, len(images), batch):
            inp = proc(images=images[i:i+batch], return_tensors='pt').to(DEVICE)
            ei.append(l2(mdl.vision_model(pixel_values=inp['pixel_values']).pooler_output).cpu().numpy())
        for i in range(0, len(texts), batch):
            inp = proc(text=texts[i:i+batch], return_tensors='pt', padding='max_length', truncation=True, max_length=64).to(DEVICE)
            et.append(l2(mdl.text_model(**{k: v for k, v in inp.items() if k in ('input_ids', 'attention_mask', 'position_ids')}).pooler_output).cpu().numpy())
    return np.hstack([np.vstack(ei), np.vstack(et)])
print('encoders defined')

---
# Section 2. Dataset loaders and how each one uses the seed

Each function below reproduces one original notebook's data loading and writes the raw files for every seed in
`SEEDS` that is not yet on Drive. Comments inside the code mark every line where `42` became `seed`.

Expected array sizes per configuration (rows x embedding dimension):

| Tag | `E_tr` | `E_te` | Dimension |
|---|---|---|---|
| `sst2_bert` | 1100 | 250 | 768 |
| `agnews_roberta` | 1100 | 250 | 768 |
| `cifar10` | 1100 | 250 | 512 |
| `fashion_vit` | 1000 | 200 | 768 |
| `mscoco_flava` | 1200 | 260 | 1536 |
| `snlive_siglip` | 1200 | 270 | 1536 |

In [ ]:
def todo(tag): return [s for s in SEEDS if not os.path.exists(raw_path(tag, s))]

# ---- SST-2 / BERT and AG News / RoBERTa: the pool itself depends on the seed ----
def text_config(tag, hub, cfg, text_col, test_split, model):
    seeds = todo(tag)
    if not seeds: print(f'{tag}: all seeds on Drive'); return
    ds = load_dataset(hub, cfg) if cfg else load_dataset(hub)
    for seed in seeds:
        tr = ds['train'].shuffle(seed=seed).select(range(1100))        # was seed=42
        te = ds[test_split].shuffle(seed=seed).select(range(250))      # was seed=42
        save_raw(tag, seed, bert_cls(model, [x[text_col] for x in tr]), np.array(tr['label']),
                            bert_cls(model, [x[text_col] for x in te]), np.array(te['label']))

# ---- CIFAR-10 / CLIP: fixed pool, encoded once; the seed only shuffles the pool ----
def cifar10_clip():
    seeds = todo('cifar10')
    if not seeds: print('cifar10: all seeds on Drive'); return
    ds = load_dataset('uoft-cs/cifar10'); col = 'img' if 'img' in ds['train'].column_names else 'image'
    base = {}
    for split, n_per_class in [('train', 550), ('test', 125)]:
        imgs, lbls = [], []
        for s in ds[split]:                      # same stopping rule as the original (550 per class, both splits)
            if s['label'] in [0, 1]: imgs.append(s[col]); lbls.append(s['label'])
            if lbls.count(0) >= 550 and lbls.count(1) >= 550: break
        c0 = [i for i, l in enumerate(lbls) if l == 0][:n_per_class]
        c1 = [i for i, l in enumerate(lbls) if l == 1][:n_per_class]
        order = c0 + c1                          # the original's `combined` list before shuffling
        base[split] = (clip_images([imgs[i] for i in order]), np.array([lbls[i] for i in order]))
    for seed in seeds:
        # The original calls random.seed(42) once, then random.shuffle(train combined), then random.shuffle(test combined).
        # random.shuffle depends only on list length, so shuffling index lists gives the same permutations.
        random.seed(seed)
        p_tr = list(range(len(base['train'][1]))); random.shuffle(p_tr)
        p_te = list(range(len(base['test'][1])));  random.shuffle(p_te)
        save_raw('cifar10', seed, base['train'][0][p_tr], base['train'][1][p_tr], base['test'][0][p_te], base['test'][1][p_te])

# ---- Fashion-MNIST / ViT: the pool depends on the seed ----
def balanced_indices(labels, a, b, n_total, seed):
    labels = np.asarray(labels); rng = np.random.RandomState(seed)
    ia = np.where(labels == a)[0]; rng.shuffle(ia); ib = np.where(labels == b)[0]; rng.shuffle(ib)
    idx = np.concatenate([ia[:n_total // 2], ib[:n_total // 2]]); rng.shuffle(idx); return idx

def fashion_vit():
    seeds = todo('fashion_vit')
    if not seeds: print('fashion_vit: all seeds on Drive'); return
    ds = load_dataset('zalando-datasets/fashion_mnist'); col = 'image' if 'image' in ds['train'].column_names else 'img'
    for seed in seeds:
        tr = ds['train'].select(balanced_indices(ds['train']['label'], 0, 1, 1000, seed))
        te = ds['test'].select(balanced_indices(ds['test']['label'], 0, 1, 200, seed))
        save_raw('fashion_vit', seed, vit_images(tr[col]), np.array(tr['label']), vit_images(te[col]), np.array(te['label']))

# ---- MSCOCO / FLAVA: fixed image pool, towers encoded once; the seed only re-pairs captions ----
def mscoco_flava():
    seeds = todo('mscoco_flava')
    if not seeds: print('mscoco_flava: all seeds on Drive'); return
    all_imgs, all_caps = [], []
    try:
        ds = load_dataset('nlphuji/mscoco_2014_5k_test_image_text_retrieval', split='test'); cols = ds.column_names
        ic = 'image' if 'image' in cols else [c for c in cols if 'image' in c.lower()][0]
        cc = 'caption' if 'caption' in cols else ('captions' if 'captions' in cols else [c for c in cols if 'caption' in c.lower()][0])
        for s in ds:
            cap = s[cc]; cap = (cap[0] if len(cap) else '') if isinstance(cap, list) else cap
            img = s[ic]
            if not isinstance(img, Image.Image): continue
            all_imgs.append(img.convert('RGB')); all_caps.append(str(cap))
            if len(all_imgs) >= 780: break
    except Exception as e:
        print(f'  nlphuji failed ({type(e).__name__}); lmms-lab fallback')
        for s in load_dataset('lmms-lab/COCO-Caption', split='val', streaming=True):
            img = s.get('image'); cap = s.get('answer')
            if isinstance(cap, list) and len(cap): cap = cap[0]
            if not isinstance(img, Image.Image): continue
            all_imgs.append(img.convert('RGB')); all_caps.append(str(cap))
            if len(all_imgs) >= 780: break
    img_cache, txt_cache = flava_towers(all_imgs, all_caps)
    for seed in seeds:
        rng = np.random.RandomState(seed)                                # was RandomState(42)
        def build_pairs(idxs):
            pairs = []
            for i in idxs:
                pairs.append((i, i, 1)); j = rng.choice([k for k in idxs if k != i]); pairs.append((i, int(j), 0))
            rng.shuffle(pairs); return pairs
        tp, sp = build_pairs(list(range(600))), build_pairs(list(range(600, 730)))
        asm = lambda P: (np.array([np.concatenate([img_cache[a], txt_cache[c]]) for a, c, _ in P]), np.array([l for _, _, l in P]))
        (Etr, ytr), (Ete, yte) = asm(tp), asm(sp)
        save_raw('mscoco_flava', seed, Etr, ytr, Ete, yte)

# ---- SNLI-VE / SigLIP: fixed pool, encoded once; identical raw file for every seed ----
def snlive_siglip():
    seeds = todo('snlive_siglip')
    if not seeds: print('snlive_siglip: all seeds on Drive'); return
    ds = load_dataset('sedrickkeh/snli-ve'); cols = ds['train'].column_names
    ic = 'image' if 'image' in cols else [c for c in cols if 'image' in c.lower() or 'img' in c.lower()][0]
    tc = 'hypothesis' if 'hypothesis' in cols else [c for c in cols if 'hypo' in c.lower() or 'sentence' in c.lower()][0]
    lc = 'label' if 'label' in cols else [c for c in cols if 'label' in c.lower() or 'gold' in c.lower()][0]
    S2I = {'entailment': 0, 'neutral': 1, 'contradiction': 2}
    gl = lambda s: S2I.get(s[lc], s[lc]) if isinstance(s[lc], str) else s[lc]
    def load_balanced(split, n):
        I, T, L = [], [], []; cnt = {0: 0, 1: 0, 2: 0}
        for s in ds[split]:
            l = gl(s)
            if l not in [0, 1, 2] or cnt[l] >= n: continue
            img = s[ic]
            if not isinstance(img, Image.Image): continue
            I.append(img.convert('RGB')); T.append(s[tc]); L.append(l); cnt[l] += 1
            if all(v >= n for v in cnt.values()): break
        return I, T, np.array(L)
    tr_i, tr_t, tr_l = load_balanced('train', 400)
    te_i, te_t, te_l = load_balanced('test' if 'test' in ds else 'validation', 90)
    E_tr, E_te = siglip_towers(tr_i, tr_t), siglip_towers(te_i, te_t)
    for seed in seeds: save_raw('snlive_siglip', seed, E_tr, tr_l, E_te, te_l)
print('extractors defined')

---
# Section 3. Run all configurations

Resumable. Each configuration prints the files it saves, with their shapes, or `all seeds on Drive` if nothing is
left to do. Expect roughly 2 to 5 minutes for the text configurations, 5 to 10 for the vision ones, and 10 to 20
each for MSCOCO and SNLI-VE, mostly download time.

In [ ]:
JOBS = [('sst2_bert',      lambda: text_config('sst2_bert', 'nyu-mll/glue', 'sst2', 'sentence', 'validation', 'bert-base-uncased')),
        ('agnews_roberta', lambda: text_config('agnews_roberta', 'fancyzhx/ag_news', None, 'text', 'test', 'roberta-base')),
        ('cifar10',        cifar10_clip),
        ('fashion_vit',    fashion_vit),
        ('mscoco_flava',   mscoco_flava),
        ('snlive_siglip',  snlive_siglip)]
for tag, job in JOBS:
    t0 = time.time(); print(f'\n=== {tag} ===')
    job()
    print(f'  {tag} done in {(time.time()-t0)/60:.1f} min')
print('\nSection 3 finished')

---
# Section 4. Inventory

Lists every raw file this study needs, with its shapes. All 18 lines should read `ok` before moving to Notebook B.

In [ ]:
EXPECTED = {'sst2_bert': (1100, 250, 768), 'agnews_roberta': (1100, 250, 768), 'cifar10': (1100, 250, 512),
            'fashion_vit': (1000, 200, 768), 'mscoco_flava': (1200, 260, 1536), 'snlive_siglip': (1200, 270, 1536)}
ok = 0
for tag, (ntr, nte, d) in EXPECTED.items():
    for seed in SEEDS:
        p = raw_path(tag, seed)
        if not os.path.exists(p): print(f'{tag:<16} seed {seed:<4} MISSING'); continue
        z = np.load(p); shapes = (z['E_tr'].shape, z['E_te'].shape)
        good = shapes == ((ntr, d), (nte, d)) and len(z['y_tr']) == ntr and len(z['y_te']) == nte
        ok += good
        print(f'{tag:<16} seed {seed:<4} {"ok" if good else "UNEXPECTED SHAPE"}  E_tr {shapes[0]}  E_te {shapes[1]}')
print(f'\n{ok} of 18 files present with the expected shapes')

---
# Section 5. Troubleshooting and hand-off

**Image embeddings do not reproduce in Notebook B.** This notebook forces exact fp32 (no TF32) and the slow,
PIL-based image processors (`use_fast=False`), because those are what the original run used. If Notebook B still
reports FAIL for an image configuration, confirm this notebook printed `TF32 off` at setup and that the `use_fast`
arguments were accepted without a warning. Put the failing tags in `REDO` in Section 0 and rerun Section 3.

**A dataset fails to download.** Hugging Face sometimes rate-limits anonymous requests. Rerun the cell; files
already saved are skipped. MSCOCO has an automatic fallback source (`lmms-lab/COCO-Caption`), and if it is used the
first 730 images may differ from the original run, which Notebook B's seed-42 check will detect.

**The session times out.** Reconnect, rerun Section 0, then Section 3. Nothing already on Drive is recomputed.

**`UNEXPECTED SHAPE` in Section 4.** The dataset on the Hub has changed since the main run (fewer examples of a class,
a different column). Report the tag; do not use that file.

**Hand-off.** When Section 4 shows 18 of 18, open `seeds_B_kernels.ipynb` on an ordinary (CPU) runtime and run it
top to bottom. Its Section 3 verifies seed 42 against the original run before any new result is produced.